## Visualisation of Co-activation Patterns (CAPs) results with Atlas Reader

Atlasreader is a Python interface that generates coordinate tables and region labels from statistical MRI images. 

Input:
- 'Spm T file' of the desired contrast
- Output specifications (direction, cluster extent, voxel threshold, type of plots, atlas)

Output:
- Overview figure showing the results within the whole brain at once
- Informative figure for each cluster showing the sagittal, coronal and transverasl plane centered on the main peak of the cluster.
- A csv file containing relevant information about the peak of each cluster.
- A csv file containing relevant information about each cluster. Table showing relevant information for the cluster extent of each ROI.

In [1]:
#Import package
from atlasreader import create_output
from IPython.display import display, Image
import os
import re
import pandas as pd
import nibabel as nib
from scipy.ndimage import affine_transform
import numpy as np

The Python package you are importing, AtlasReader, is licensed under the
BSD-3 license; however, the atlases it uses are separately licensed under more
restrictive frameworks.
By using AtlasReader, you agree to abide by the license terms of the
individual atlases. Information on these terms can be found online at:
https://github.com/miykael/atlasreader/tree/master/atlasreader/data



Additional step to Confirm and Permanently Fix Negative pixdim Values.

Without this following step following error message was received:
pixdim[1,2,3] should be positive; setting to abs of pixdim values
pixdim values: [1. 2. 2. 2. 1. 1. 1. 1.]


In [ ]:
# Define root directory and a single output directory
root_dir = ''
output_root_dir = os.path.join(root_dir, 'AtlasReader', '')

# Ensure the output directory exists
os.makedirs(output_root_dir, exist_ok=True)

# Regular expression to match files with the naming pattern 
file_pattern = re.compile(r'^\d+\.nii$')

# Loop through files in the directory
for file_name in os.listdir(root_dir):
    # Check if the file matches the pattern
    if file_pattern.match(file_name):
        file_dir = os.path.join(root_dir, file_name)
        
        # Set the output path as a single directory for all files
        output_dir = output_root_dir
        
        # Print progress
        print(f"Processing file: {file_name}")
        
        # Create output for each file
        create_output(
            file_dir,
            cluster_extent=20, 
            direction="both",  # Show both positive and negative thresholds
            outdir=output_dir, 
            voxel_thresh=1.65,  
            glass_plot_kws={"black_bg": False}, 
            stat_plot_kws={"black_bg": False, "title": f"Z-score Threshold ±1.65 for {file_name}"},
            atlas=["harvard_oxford", "AAL", "juelich"]
        )
        
        print(f"Output saved for file: {file_name} in {output_dir}")

print("Processing complete for all matching files.")

After execution, we get four different kind of files:

#### Overview figure
An overview figure that shows the results within the whole brain at once
For each cluster, an informative figure showing the sagittal, coronal and trasnveral plane centered at this center of the cluster
A csv file containing relevant information about the peak of each cluster
A csv file containing relevant information about each cluster

In [ ]:
# Get the name of the overview figure and display 

# Loop through files in the directory
for file_name in os.listdir(root_dir):
    # Check if the file matches the pattern
    if file_pattern.match(file_name):
        file_dir = os.path.join(root_dir, file_name)
# Split the filename into name and extension
        name, ext = os.path.splitext(file_name)

# Change the extension to .png
        image_name = f"{name}.png"
        image_dir = os.path.join(output_dir, image_name)

       # Display the image if it exists
        if os.path.exists(image_dir):
            display(Image(image_dir))
            print(f"Displayed overview image for {file_name}")
        else:
            print(f"Overview image not found for {file_name}")

In [ ]:
# Loop through files to read and display each peak table
for file_name in os.listdir(root_dir):
    # Check if the file matches the pattern
    if file_pattern.match(file_name):
        # Define path to the peak table CSV file
        name, ext = os.path.splitext(file_name)
        csv_name = f"{name}_peaks.csv"  # Assume naming convention for CSV
        peak_csv_dir = os.path.join(output_root_dir, csv_name)  # Full path to the CSV file

        # Read and display the CSV file if it exists
        if os.path.exists(peak_csv_dir):
            peak_table = pd.read_csv(peak_csv_dir)
            print(f"\nPeak Table for {file_name}:")
            display(peak_table)  # Use display() to show a nicely formatted table
        else:
            print(f"Peak table CSV not found for {file_name}")

In [ ]:
# Loop through files to read and display each cluster table
for file_name in os.listdir(root_dir):
    # Check if the file matches the pattern
    if file_pattern.match(file_name):
        # Define path to the cluster table CSV file
        name, ext = os.path.splitext(file_name)
        csv_name = f"{name}_clusters.csv"  # Assume naming convention for clusters CSV
        clusters_csv_dir = os.path.join(output_root_dir, csv_name)  # Full path to the CSV file

        # Read and display the CSV file if it exists
        if os.path.exists(clusters_csv_dir):
            cluster_table = pd.read_csv(clusters_csv_dir)
            print(f"\nCluster Table for {file_name}:")
            display(cluster_table)  # Use display() to show a nicely formatted table
        else:
            print(f"Cluster table CSV not found for {file_name}")

In [ ]:
# Loop through files to read and display each cluster table - divide between co-activation and co-deactivation
for file_name in os.listdir(root_dir):
    # Check if the file matches the pattern
    if file_pattern.match(file_name):
        # Define path to the cluster table CSV file
        name, ext = os.path.splitext(file_name)
        csv_name = f"{name}_peaks.csv"  # Assume naming convention for clusters CSV
        peak_csv_dir = os.path.join(output_root_dir, csv_name)  # Full path to the CSV file

        # Read and display the CSV file if it exists
        if os.path.exists(peak_csv_dir):
            peak_table = pd.read_csv(peak_csv_dir)
             # Add Activation Type column
            if 'peak_value' in peak_table.columns:
                peak_table['Activation_Type'] = np.where(
                    peak_table['peak_value'] > 0, 'Co-activation',
                    np.where(peak_table['peak_value'] < 0, 'Co-deactivation', 'Neutral')
                )
            print(f"\nCluster Table for {file_name}:")
            display(peak_table)  # Use display() to show a nicely formatted table
                        # Save labeled table
            labeled_csv_name = f"{name}_peak_labeled.csv"
            labeled_csv_path = os.path.join(output_root_dir, labeled_csv_name)
            cluster_table.to_csv(labeled_csv_path, index=False)
        else:
            print(f"Cluster table CSV not found for {file_name}")

In [ ]:
# Loop through files to read and display each cluster table - divide between co-activation and co-deactivation
for file_name in os.listdir(root_dir):
    # Check if the file matches the pattern
    if file_pattern.match(file_name):
        # Define path to the cluster table CSV file
        name, ext = os.path.splitext(file_name)
        csv_name = f"{name}_clusters.csv"  # Assume naming convention for clusters CSV
        clusters_csv_dir = os.path.join(output_root_dir, csv_name)  # Full path to the CSV file

        # Read and display the CSV file if it exists
        if os.path.exists(clusters_csv_dir):
            cluster_table = pd.read_csv(clusters_csv_dir)
             # Add Activation Type column
            if 'cluster_mean' in cluster_table.columns:
                cluster_table['Activation_Type'] = np.where(
                    cluster_table['cluster_mean'] > 0, 'Co-activation',
                    np.where(cluster_table['cluster_mean'] < 0, 'Co-deactivation', 'Neutral')
                )
            print(f"\nCluster Table for {file_name}:")
            display(cluster_table)  # Use display() to show a nicely formatted table
                        # Save labeled table
            labeled_csv_name = f"{name}_cluster_labeled.csv"
            labeled_csv_path = os.path.join(output_root_dir, labeled_csv_name)
            cluster_table.to_csv(labeled_csv_path, index=False)
        else:
            print(f"Cluster table CSV not found for {file_name}")